In [ ]:
import joblib
import json
from pathlib import Path

MODEL_DIR = Path("../models")  # adjust only if your notebook is elsewhere

model = joblib.load(MODEL_DIR / "premier_league_model.joblib")

with open(MODEL_DIR / "feature_schema.json", "r") as f:
    feature_schema = json.load(f)

with open(MODEL_DIR / "model_metadata.json", "r") as f:
    metadata = json.load(f)

print("Model type:", type(model).__name__)
print("Number of schema features:", len(feature_schema))
print("Classes:", model.classes_)
print("\nMetadata:")
print(metadata)

In [ ]:
print("Model expects:", model.n_features_in_)
print("Schema contains:", len(feature_schema))

if hasattr(model, "feature_names_in_"):
    model_features = list(model.feature_names_in_)

    print("Model has stored feature names:", len(model_features))
    print("Exact same order:", model_features == feature_schema)

    if model_features != feature_schema:
        for i, (model_col, schema_col) in enumerate(
            zip(model_features, feature_schema), start=1
        ):
            if model_col != schema_col:
                print(
                    f"Mismatch {i}: "
                    f"model={model_col} | schema={schema_col}"
                )

In [ ]:
import pandas as pd
import numpy as np

PROCESSED_DIR = Path("../data/processed")

df = pd.read_csv(PROCESSED_DIR / "model_features.csv")

print("Rows:", len(df))
print("Schema features available:", sum(c in df.columns for c in feature_schema))

missing_columns = [
    c for c in feature_schema
    if c not in df.columns
]

print("Missing columns:", missing_columns)

complete = df.dropna(subset=feature_schema).copy()

print("Complete model rows:", len(complete))

In [ ]:
test = complete[
    complete["season"] == "2025_26"
].copy()

test = test.sort_values("date")

X_test = test[feature_schema]

print("Test matches:", len(test))

probs = model.predict_proba(X_test)

print("Prediction shape:", probs.shape)
print("First prediction:", probs[0])
print(
    "Probability sums valid:",
    np.allclose(probs.sum(axis=1), 1.0)
)

In [ ]:
print(test["winner"].value_counts())
print(test["winner"].unique())

In [ ]:
from sklearn.metrics import log_loss, accuracy_score
import numpy as np

# Explicit class mapping
winner_map = {
    "home": 0,
    "draw": 1,
    "away": 2
}

y_test = test["winner"].map(winner_map)

# Safety checks
assert y_test.notna().all()
assert list(model.classes_) == [0, 1, 2]
assert list(X_test.columns) == feature_schema
assert len(X_test) == 357

# Predictions
probs = model.predict_proba(X_test)
preds = model.predict(X_test)

# Metrics
ll = log_loss(
    y_test,
    probs,
    labels=[0, 1, 2]
)

acc = accuracy_score(
    y_test,
    preds
)

print("=== FINAL MODEL VERIFICATION ===")
print("Model:", type(model).__name__)
print("Matches:", len(X_test))
print("Features:", X_test.shape[1])
print("Classes:", model.classes_)
print("Feature order exact:", list(X_test.columns) == feature_schema)
print("Probabilities valid:", np.allclose(probs.sum(axis=1), 1.0))
print()
print(f"Log Loss: {ll:.6f}")
print(f"Accuracy: {acc:.4f}")
print()
print("Expected packaged log loss:", metadata["log_loss"])
print("Difference:", abs(ll - metadata["log_loss"]))

In [ ]:
# ============================================================
# PRODUCTION MODEL DATA AUDIT
# Does every match actually have every feature the model needs?
# ============================================================

import pandas as pd
import numpy as np

print("=== PRODUCTION DATA AUDIT ===")

print("\nTotal complete model rows:", len(complete))

# 1. Exact 51 production features
print("\n--- FEATURE COMPLETENESS ---")

missing_counts = complete[feature_schema].isna().sum()
inf_counts = pd.Series(
    np.isinf(complete[feature_schema].select_dtypes(include=np.number)).sum(),
    name="inf_count"
)

print("Missing values across 51 features:", int(missing_counts.sum()))
print("Infinite values:", int(inf_counts.sum()))

bad_features = missing_counts[missing_counts > 0]

if len(bad_features) == 0:
    print("✓ All 51 production features are populated for every model row.")
else:
    print("Features with missing values:")
    print(bad_features)


# 2. Check the underlying match statistics we originally cared about
raw_stats = [
    "home_goals",
    "away_goals",
    "home_xg",
    "away_xg",
    "home_shots",
    "away_shots",
    "home_shots_on_target",
    "away_shots_on_target",
    "home_possession",
    "away_possession",
    "home_passes",
    "away_passes",
    "home_corners",
    "away_corners",
]

available_raw_stats = [
    c for c in raw_stats
    if c in complete.columns
]

print("\n--- RAW MATCH STAT COMPLETENESS ---")

for col in raw_stats:

    if col not in complete.columns:
        print(f"{col:35} NOT PRESENT")
        continue

    missing = complete[col].isna().sum()

    print(
        f"{col:35} "
        f"missing={missing:4} "
        f"complete={len(complete)-missing:4}/{len(complete)}"
    )


# 3. Break it down by season
print("\n--- COMPLETE MODEL ROWS BY SEASON ---")

print(
    complete["season"]
    .value_counts()
    .sort_index()
)


# 4. Specifically audit the 357-match test set
print("\n--- 2025/26 TEST SET ---")

print("Matches:", len(test))

test_feature_missing = test[feature_schema].isna().sum().sum()

print(
    "Missing values across 51 production features:",
    test_feature_missing
)

for col in raw_stats:

    if col in test.columns:
        print(
            f"{col:35} "
            f"missing={test[col].isna().sum()}"
        )


# 5. Find any suspicious rows
check_cols = [
    c for c in raw_stats
    if c in complete.columns
]

problem_rows = complete[
    complete[check_cols].isna().any(axis=1)
]

print("\n--- MATCHES WITH ANY MISSING RAW STATS ---")
print("Problem matches:", len(problem_rows))

if len(problem_rows) > 0:

    display_cols = [
        c for c in [
            "date",
            "season",
            "home_team",
            "away_team",
            *check_cols
        ]
        if c in problem_rows.columns
    ]

    print(
        problem_rows[display_cols]
        .to_string(index=False)
    )
else:
    print("✓ No model rows contain missing raw match statistics.")